# Evaluation: DenseNet201 + Augmentation
## Evaluation for model with Augmentation (No FPN, No Class Weighting, No GCA)

This notebook evaluates the model trained with augmentation.

In [ ]:
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple, Optional
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Torchvision imports
from torchvision import transforms
from torchvision.models import densenet201, DenseNet201_Weights

# Albumentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Scikit-learn metrics
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    roc_auc_score
)

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️  Using device: {DEVICE}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# CONFIGURATION
class Config:
    """Evaluation configuration."""
    
    # Paths
    MODEL_PATH = Path("outputs_augmentation/final_model.pth")
    DATA_ROOT = Path("../dataset/Raw_dataset")
    OUTPUT_DIR = Path("evaluation_augmentation")
    
    # Dataset parameters
    CLASSES = ['Healthy', 'Anthracnose']
    NUM_CLASSES = 2
    PLANTS = ['Guava', 'Mango', 'Papaya']
    
    # Image parameters
    IMG_SIZE = 224
    IMG_MEAN = [0.485, 0.456, 0.406]
    IMG_STD = [0.229, 0.224, 0.225]
    
    # Evaluation parameters
    BATCH_SIZE = 24
    DROPOUT_RATE = 0.3
    
    @classmethod
    def create_dirs(cls):
        cls.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        print("✅ Output directory created")

Config.create_dirs()
print(f"\n📋 Evaluation Configuration:")
print(f"   Model Path: {Config.MODEL_PATH}")
print(f"   Data Root: {Config.DATA_ROOT}")
print(f"   Output Dir: {Config.OUTPUT_DIR}")

In [ ]:
class DenseNetWithAugmentation(nn.Module):
    """DenseNet201 classifier - NO FPN, NO GCA."""
    
    def __init__(self, num_classes: int = 2, pretrained: bool = True, dropout_rate: float = 0.3):
        super().__init__()
        
        # Load pretrained DenseNet201
        if pretrained:
            weights = DenseNet201_Weights.IMAGENET1K_V1
            self.backbone = densenet201(weights=weights)
        else:
            self.backbone = densenet201(weights=None)
        
        # Get number of features from backbone
        num_features = self.backbone.classifier.in_features
        
        # Replace classifier with simple head
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate),
            nn.Linear(num_features, num_classes)
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.backbone(x)

print("✅ Model class defined")

In [ ]:
# Load trained model
model = DenseNetWithAugmentation(
    num_classes=Config.NUM_CLASSES,
    pretrained=False,  # Will load trained weights
    dropout_rate=Config.DROPOUT_RATE
).to(DEVICE)

# Load checkpoint
checkpoint = torch.load(Config.MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print("✅ Model loaded successfully")
print(f"   Model: DenseNet201 + Augmentation")
print(f"   Checkpoint timestamp: {checkpoint.get('timestamp', 'N/A')}")
if 'final_metrics' in checkpoint:
    print(f"   Best F1 Score: {checkpoint['final_metrics']['best_f1']:.4f}")
    print(f"   Best Epoch: {checkpoint['final_metrics']['best_epoch']}")

In [ ]:
# Load dataset
from sklearn.model_selection import train_test_split

def load_dataset(data_root: Path) -> pd.DataFrame:
    """Load dataset from directory structure."""
    data = []
    
    for plant in Config.PLANTS:
        for class_name in Config.CLASSES:
            folder = data_root / plant / class_name
            if not folder.exists():
                continue
            
            for img_path in folder.glob('*.jpg'):
                data.append({
                    'path': str(img_path),
                    'plant': plant,
                    'class': class_name,
                    'label': Config.CLASSES.index(class_name)
                })
    
    return pd.DataFrame(data)

# Load and split data
df = load_dataset(Config.DATA_ROOT)

train_df, temp_df = train_test_split(
    df, test_size=0.3, stratify=df['label'], random_state=42
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42
)

print(f"✅ Dataset loaded:")
print(f"   Total: {len(df)} images")
print(f"   Test: {len(test_df)} images")
print(f"\n📊 Test Set Distribution:")
print(test_df['class'].value_counts())

In [ ]:
# Test transforms (no augmentation for evaluation)
test_transforms = A.Compose([
    A.Resize(Config.IMG_SIZE, Config.IMG_SIZE),
    A.Normalize(mean=Config.IMG_MEAN, std=Config.IMG_STD),
    ToTensorV2()
])

class AugmentedAnthracnoseDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transforms=None):
        self.data = dataframe.reset_index(drop=True)
        self.transforms = transforms
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['path']).convert('RGB')
        image = np.array(image)
        
        if self.transforms:
            augmented = self.transforms(image=image)
            image = augmented['image']
        
        return {
            'image': image,
            'label': row['label'],
            'path': row['path'],
            'plant': row['plant']
        }

# Create test dataset and loader
test_dataset = AugmentedAnthracnoseDataset(test_df, transforms=test_transforms)
test_loader = DataLoader(
    test_dataset,
    batch_size=Config.BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(f"✅ Test dataset created: {len(test_dataset)} samples")
print(f"   Test batches: {len(test_loader)}")

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader, device):
    """Comprehensive model evaluation."""
    model.eval()
    
    all_preds = []
    all_labels = []
    all_probs = []
    all_plants = []
    
    print("\n🔍 Evaluating model...")
    for batch in tqdm(dataloader, desc="Evaluation"):
        images = batch['image'].to(device)
        labels = batch['label'].to(device)
        plants = batch['plant']
        
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        preds = outputs.argmax(dim=1)
        
        all_probs.extend(probs.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_plants.extend(plants)
    
    all_probs = np.array(all_probs)
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    results = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='weighted'),
        'recall': recall_score(all_labels, all_preds, average='weighted'),
        'f1': f1_score(all_labels, all_preds, average='weighted'),
        'auc': roc_auc_score(all_labels, all_probs[:, 1]),
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'plants': all_plants
    }
    
    # Classification report
    results['classification_report'] = classification_report(
        all_labels, all_preds, 
        target_names=Config.CLASSES,
        digits=4
    )
    
    # Confusion matrix
    results['confusion_matrix'] = confusion_matrix(all_labels, all_preds)
    
    return results

# Run evaluation
results = evaluate_model(model, test_loader, DEVICE)

print("\n" + "="*60)
print("📊 EVALUATION RESULTS (DenseNet + Augmentation)")
print("="*60)
print(f"\n🎯 Overall Metrics:")
print(f"   Accuracy:  {results['accuracy']:.4f}")
print(f"   Precision: {results['precision']:.4f}")
print(f"   Recall:    {results['recall']:.4f}")
print(f"   F1 Score:  {results['f1']:.4f}")
print(f"   AUC:       {results['auc']:.4f}")

print(f"\n📋 Classification Report:")
print(results['classification_report'])

In [ ]:
# Plot confusion matrix
def plot_confusion_matrix(cm, class_names, save_path=None):
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={'label': 'Count'}
    )
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.title('Confusion Matrix - DenseNet + Augmentation')
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ Saved to {save_path}")
    
    plt.show()

plot_confusion_matrix(
    results['confusion_matrix'],
    Config.CLASSES,
    Config.OUTPUT_DIR / 'confusion_matrix.png'
)

In [ ]:
# Plot ROC curve
def plot_roc_curve(labels, probs, save_path=None):
    fpr, tpr, _ = roc_curve(labels, probs[:, 1])
    roc_auc = auc(fpr, tpr)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - DenseNet + Augmentation')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"✅ Saved to {save_path}")
    
    plt.show()

plot_roc_curve(
    results['labels'],
    results['probabilities'],
    Config.OUTPUT_DIR / 'roc_curve.png'
)

In [ ]:
# Per-plant performance analysis
def analyze_per_plant_performance(predictions, labels, plants):
    """Analyze performance separately for each plant type."""
    results = {}
    
    print("\n" + "="*60)
    print("🌿 PER-PLANT PERFORMANCE ANALYSIS")
    print("="*60)
    
    for plant in Config.PLANTS:
        mask = np.array([p == plant for p in plants])
        plant_preds = predictions[mask]
        plant_labels = labels[mask]
        
        if len(plant_labels) == 0:
            continue
        
        acc = accuracy_score(plant_labels, plant_preds)
        f1 = f1_score(plant_labels, plant_preds, average='weighted')
        
        results[plant] = {'accuracy': acc, 'f1': f1}
        
        print(f"\n{plant}:")
        print(f"   Samples: {len(plant_labels)}")
        print(f"   Accuracy: {acc:.4f}")
        print(f"   F1 Score: {f1:.4f}")
    
    return results

per_plant_results = analyze_per_plant_performance(
    results['predictions'],
    results['labels'],
    results['plants']
)

In [ ]:
# Save evaluation results
eval_summary = {
    'model': 'DenseNet201_with_Augmentation',
    'features': 'Augmentation ENABLED, No FPN, No Class Weighting, No GCA',
    'overall_metrics': {
        'accuracy': float(results['accuracy']),
        'precision': float(results['precision']),
        'recall': float(results['recall']),
        'f1': float(results['f1']),
        'auc': float(results['auc'])
    },
    'per_plant_metrics': {
        plant: {
            'accuracy': float(metrics['accuracy']),
            'f1': float(metrics['f1'])
        }
        for plant, metrics in per_plant_results.items()
    },
    'confusion_matrix': results['confusion_matrix'].tolist(),
    'test_samples': len(test_df)
}

with open(Config.OUTPUT_DIR / 'evaluation_results.json', 'w') as f:
    json.dump(eval_summary, f, indent=2)

print(f"\n✅ Evaluation results saved to {Config.OUTPUT_DIR / 'evaluation_results.json'}")

print("\n" + "="*60)
print("✅ EVALUATION COMPLETE")
print("="*60)

In [ ]:
# Visualize sample predictions
def visualize_predictions(model, dataset, num_samples=8, device='cuda'):
    """Visualize sample predictions."""
    model.eval()
    
    # Get random samples
    indices = np.random.choice(len(dataset), num_samples, replace=False)
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.flatten()
    
    with torch.no_grad():
        for idx, ax in zip(indices, axes):
            sample = dataset[idx]
            image = sample['image'].unsqueeze(0).to(device)
            true_label = sample['label']
            plant = sample['plant']
            
            # Predict
            output = model(image)
            prob = F.softmax(output, dim=1)
            pred_label = output.argmax(dim=1).item()
            confidence = prob[0, pred_label].item()
            
            # Display image
            img_np = image[0].cpu().permute(1, 2, 0).numpy()
            img_np = img_np * np.array(Config.IMG_STD) + np.array(Config.IMG_MEAN)
            img_np = np.clip(img_np, 0, 1)
            
            ax.imshow(img_np)
            ax.axis('off')
            
            # Title with prediction
            true_class = Config.CLASSES[true_label]
            pred_class = Config.CLASSES[pred_label]
            color = 'green' if pred_label == true_label else 'red'
            
            ax.set_title(
                f'{plant}\nTrue: {true_class}\nPred: {pred_class} ({confidence:.2f})',
                color=color,
                fontsize=10
            )
    
    plt.tight_layout()
    plt.savefig(Config.OUTPUT_DIR / 'sample_predictions.png', dpi=150, bbox_inches='tight')
    print(f"✅ Sample predictions saved")
    plt.show()

visualize_predictions(model, test_dataset, num_samples=8, device=DEVICE)

In [ ]:
# Optional: Compare with baseline results
print("\n" + "="*60)
print("📊 COMPARISON NOTES")
print("="*60)
print("\nThis model includes data augmentation.")
print("Compare with baseline (no augmentation) to measure:")
print("  - Impact of geometric transforms (rotation, flip, shift)")
print("  - Impact of color augmentations (brightness, contrast, hue)")
print("  - Impact of noise and blur augmentations")
print("\nExpected improvement: 2-5% in accuracy and F1 score")
print("Better generalization and robustness to variations")